# Parallel analysis with entarchy

`Collection.map_async` applies a function to every entity in a collection using
worker processes. This notebook shows how to use it, and the one thing to watch
out for when working in a notebook.

## Functions must be importable

Worker processes are started with the `spawn` method, so they have to import the
function being mapped. A function defined in a notebook cell lives in `__main__`,
which the workers cannot import.

entarchy detects this. If [cloudpickle](https://pypi.org/project/cloudpickle/) is
installed, the definition is sent to the workers by value and everything works. If
it is not, entarchy prints a warning and runs the work in this process rather than
hanging.

For production pipelines, put analysis functions in a module and import them: it
is faster, and it keeps the analysis under version control.

In [ ]:
import os
import shutil
import tempfile

import numpy as np

import entarchy
from entarchy.backend import SQLiteBackend
from entarchy.core.entity import shutdown_worker_pool

In [ ]:
class Recording(entarchy.Entity):
    pass


class Roi(entarchy.Entity):
    pass


Recording.add_child_entity_type(Roi)


class Demo(entarchy.Entarchy):
    _implementation_version = '0.1'
    _implementation_compat_version_list = ['0.1']
    _hierarchy_root_type = Recording


path = os.path.join(tempfile.mkdtemp(), 'parallel_demo')
ent = Demo.create(path, SQLiteBackend(path, dbname='demo.db'))

rng = np.random.default_rng(1)
with ent:
    for rec_index in range(2):
        recording = Recording(ent, _id=f'rec_{rec_index}', _parent=ent.root)
        ent.add_new_entity(recording)
        recording['imaging_rate'] = 10.0

        for roi_index in range(50):
            roi = Roi(ent, _id=f'roi_{roi_index}', _parent=recording)
            ent.add_new_entity(roi)
            roi['index'] = roi_index
            roi['trace'] = rng.normal(loc=100.0, scale=5.0, size=500)

ent.get(Roi)

## Mapping a function

The function receives one entity and writes its results back onto it. Anything it
returns is ignored by `map_async`, so results must be stored on the entity.

In [ ]:
def compute_snr(roi):
    """Signal to noise ratio of a trace. Defined in a cell, which is the case
    entarchy has to handle carefully."""
    trace = roi['trace']
    roi['snr'] = float(trace.mean() / trace.std())


ent.get(Roi).map_async(compute_snr, _worker_num=2)

In [ ]:
ent.get(Roi).preview(5, attribute_names=['index', 'snr'])

## When are workers worth it?

Starting a pool costs a couple of seconds, which is more than the work itself for
small or cheap jobs. `map_async` measures the first entities and skips the pool when
it would not pay off, so it is never slower than running in this process. Pass
`_calibrate=False` to force the pool.

Workers stay alive between calls, so a sequence of analysis steps pays the startup
cost once. Release them explicitly when finished.

In [ ]:
def compute_baseline(roi):
    trace = roi['trace']
    roi['baseline'] = float(np.percentile(trace, 10))


# Whichever call first starts a pool, later calls reuse it: watch for
# "Start worker pool" the first time and "Reuse worker pool" afterwards
ent.get(Roi).map_async(compute_baseline, _worker_num=2, _calibrate=False)

## Failures do not abandon the run

If the function raises for some entities, the rest are still processed and
committed. A summary is printed and an error is raised at the end, so a partially
failed run can be resumed by selecting the entities that are still missing the
attribute.

In [ ]:
def sometimes_fails(roi):
    if roi['index'] % 10 == 0:
        raise ValueError('cannot process this roi')
    roi['processed'] = True


try:
    ent.get(Roi).map_async(sometimes_fails, _worker_num=2, _calibrate=False)
except RuntimeError as e:
    print(f'caught: {str(e).splitlines()[0]}')

print(f'{len(ent.get(Roi, "EXIST(processed)"))} rois processed')
print(f'{len(ent.get(Roi, "NOT(EXIST(processed))"))} rois still to do')

## Cleaning up

Worker processes hold open database connections, so release them when done.

In [ ]:
shutdown_worker_pool()
ent.backend.close()
shutil.rmtree(os.path.dirname(path), ignore_errors=True)
print('done')